# 02 ASR Baseline

Common Voice Yue-only baseline scaffold. This notebook prepares small train/dev subsets and evaluation columns without downloading or training a model yet.

In [1]:
from pathlib import Path
import os
import sys

def running_in_colab():
    return 'google.colab' in sys.modules or Path('/content').exists()

def mount_drive_if_available():
    if not running_in_colab():
        return
    try:
        from google.colab import drive
        if not Path('/content/drive').exists():
            drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {exc}')

def find_project_root():
    mount_drive_if_available()
    cwd = Path.cwd().resolve()
    env_root = os.getenv('PROJECT_ROOT')
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates += [cwd, *cwd.parents]
    candidates += [
        Path('/content/Cantonese-Speech-AI'),
        Path('/content/drive/MyDrive/Cantonese-Speech-AI'),
        Path('/content/drive/My Drive/Cantonese-Speech-AI'),
        Path(r'D:/GitHub/Cantonese-Speech-AI'),
        Path(r'D:\\GitHub\\Cantonese-Speech-AI'),
        Path('/mnt/d/GitHub/Cantonese-Speech-AI'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'cantonese_speech_ai').exists():
            return candidate.resolve()
    checked = '\n'.join(str(path) for path in candidates[:12])
    raise FileNotFoundError(
        'Cannot find the Cantonese-Speech-AI project folder in this runtime.\n\n'
        'Colab fix:\n'
        '1. Upload or copy the whole Cantonese-Speech-AI folder to Google Drive.\n'
        '2. Expected path: /content/drive/MyDrive/Cantonese-Speech-AI\n'
        '3. The folder must contain src/cantonese_speech_ai and Mozilla-HK-Speech-datasets.\n'
        '4. Then restart runtime and run this cell again.\n\n'
        'Alternative: set os.environ[\'PROJECT_ROOT\'] to your folder path before this cell.\n\n'
        f'Current cwd: {cwd}\nChecked:\n{checked}'
    )

ROOT = find_project_root()
os.environ['PROJECT_ROOT'] = str(ROOT)
default_dataset = ROOT / 'Mozilla-HK-Speech-datasets' / 'cv-corpus-26.0-2026-06-12' / 'yue'
os.environ.setdefault('COMMON_VOICE_YUE_ROOT', str(default_dataset))

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from cantonese_speech_ai.common_voice import CommonVoicePaths, prepare_split
from cantonese_speech_ai.metrics import char_error_rate, word_error_rate

paths = CommonVoicePaths.from_env()
if not paths.split_path('train').exists():
    raise FileNotFoundError(f'Dataset train.tsv not found: {paths.split_path("train")}')
ROOT, paths


(WindowsPath('D:/GitHub/Cantonese-Speech-AI'),
 CommonVoicePaths(root=WindowsPath('D:/GitHub/Cantonese-Speech-AI/Mozilla-HK-Speech-datasets/cv-corpus-26.0-2026-06-12/yue')))

## Small baseline subset

In [2]:
train = pd.DataFrame(prepare_split('train', paths)).head(500)
dev = pd.DataFrame(prepare_split('dev', paths)).head(100)
test = pd.DataFrame(prepare_split('test', paths))

baseline_columns = ['audio_path', 'sentence', 'normalized_sentence', 'duration_sec', 'accents']
train[baseline_columns].head()

,audio_path,sentence,normalized_sentence,duration_sec,accents
0,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,大陸有專俾同志用嘅交友程式？,大陸有專俾同志用嘅交友程式,4.608,广东音
1,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,仲有舊年年尾話唔應該再猛咁撩鼻，不如做抗體檢測好過，個訪問冇耐就喺大陸全網禁咗,仲有舊年年尾話唔應該再猛咁撩鼻 不如做抗體檢測好過 個訪問冇耐就喺大陸全網禁咗,10.080,广州音
2,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,邊有噉大隻蛤乸隨街跳,邊有噉大隻蛤乸隨街跳,3.528,广州音
3,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,但係我唔特登講就成日會俾人估錯隔離,但係我唔特登講就成日會俾人估錯隔離,5.148,广州音
4,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,除非有人喺第二區截車入去,除非有人喺第二區截車入去,3.636,广州音


## Model direction

Start with a pretrained Whisper or wav2vec-style model. Keep the first run small: 500 train rows and 100 dev rows. Save predictions to a CSV with `path`, `sentence`, and `prediction` columns for error analysis.

In [3]:
model_plan = {
    'dataset': 'Mozilla Common Voice Yue',
    'train_rows': len(train),
    'dev_rows': len(dev),
    'primary_metric': 'CER',
    'candidate_models': ['Whisper fine-tuning', 'wav2vec2 CTC fine-tuning'],
}
model_plan

{'dataset': 'Mozilla Common Voice Yue',
 'train_rows': 500,
 'dev_rows': 100,
 'primary_metric': 'CER',
 'candidate_models': ['Whisper fine-tuning', 'wav2vec2 CTC fine-tuning']}

## Evaluation scaffold

In [4]:
example_ref = dev.iloc[0]['normalized_sentence']
example_prediction = example_ref

{
    'cer': char_error_rate(example_ref, example_prediction),
    'wer': word_error_rate(example_ref, example_prediction),
}

{'cer': 0.0, 'wer': 0.0}